# Analyse: Körperpose


In [ ]:
import sys
!{sys.executable} -m pip install transformers torch pillow pandas numpy tqdm


In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())


c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.7.1+cpu
CUDA available: False


In [2]:
DATA_DIR = Path('../../data')
COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '11_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_body_pose.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60
MODEL_NAME = 'Harsha901/swinv2-tiny-human-pose-classification-model'
UNKNOWN_LABEL = 'Unbestimmt'

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Running on device: {device}')

df = pd.read_csv(COMBINED_CSV)


Running on device: cpu


In [3]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'

def get_video_id(row) -> str:
    value = row.get(video_id_col, None)
    if pd.isna(value):
        return ''
    return str(value)

def get_duration_seconds(row):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan

def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()

video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [4]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME).to(device).eval()
id2label = model.config.id2label

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]

def predict_pose(image_path: Path):
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt').to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
        idx = int(probs.argmax().item())
        return id2label[idx], float(probs[idx].item())
    except Exception:
        return None, np.nan


c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hanna\.cache\huggingface\hub\models--Harsha901--swinv2-tiny-human-pose-classification-model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' packa

In [5]:
pose_labels = []
pose_confs = []
pose_detected_frames = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    vid = get_video_id(row)
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(vid, duration_seconds=duration_seconds)

    labels = []
    confs = []
    for fp in frame_paths:
        lbl, conf = predict_pose(fp)
        if lbl is not None:
            labels.append(lbl)
            if not np.isnan(conf):
                confs.append(conf)

    if labels:
        pose_labels.append(Counter(labels).most_common(1)[0][0])
        pose_confs.append(float(np.mean(confs)) if confs else np.nan)
        pose_detected_frames.append(len(labels))
    else:
        pose_labels.append(UNKNOWN_LABEL)
        pose_confs.append(np.nan)
        pose_detected_frames.append(0)


Processing videos: 100%|██████████| 500/500 [36:20<00:00,  4.36s/it]  


In [6]:
df['pose_orientation'] = pose_labels
df['pose_confidence'] = pose_confs
df['detected_pose_frames'] = pose_detected_frames

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved: {OUTPUT_CSV}')
df[['pose_orientation', 'pose_confidence', 'detected_pose_frames']].head(20)


Saved: ..\..\data\04_analysis_results\visual_features\11_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_body_pose.csv


,pose_orientation,pose_confidence,detected_pose_frames
0,sitting,0.277859,7
1,laughing,0.553389,10
2,eating,0.352097,8
3,eating,0.698669,8
4,laughing,0.659677,8
5,sitting,0.524594,8
6,listening_to_music,0.615887,8
7,calling,0.700790,8
8,dancing,0.450308,6
9,laughing,0.735485,8
